# Population & Thermal Relaxation

This notebook verifies that thermal relaxation dynamics correctly enforce quantum detailed balance and drive populations toward thermodynamic equilibrium. It tests both scalar evolution matrices and full Liouvillian superoperator formulations to ensure physical consistency, probability conservation, and correct temperature scaling across different regimes.

**Specific Checks**:
1. Conservation of total probability (zero column sums in transition rate matrices)
2. Exact satisfaction of the Kubo-Martin-Schwinger (KMS) detailed balance condition for 2-level systems
3. Verification that the Boltzmann distribution is a stationary state (K·n_eq ≈ 0)
4. Correct physical scaling of transition rates with temperature (low-T downward dominance vs high-T symmetry)
5. Proper extraction and thermal scaling of the population-population block within the full Liouvillian superoperator
6. Consistency between scalar evolution matrices and full superoperator representations of thermal relaxation

In [10]:
%load_ext autoreload
%autoreload 2

import sys
import os
import math
from importlib import reload

import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import mars
from mars import spin_model, spectra_manager, constants, population, concat

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import mars.population.transform as transform
from mars.population.contexts import Context, SummedContext
from mars import spin_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from mars.population.tr_utils import EvolutionMatrix, EvolutionSuper

In [13]:
dtype = torch.float64

In [14]:
res_energies = torch.tensor([mars.constants.unit_converter(-2, "K_to_Hz"), mars.constants.unit_converter(2, "K_to_Hz")], dtype=dtype)
temp = torch.tensor(1.0, dtype=dtype)

free_probs = torch.tensor([[0.0, 10.0], [20.0, 0.0]], dtype=dtype)
driven_probs = torch.tensor([[0.0, 10.0], [20.0, 0.0]], dtype=dtype)
evo_matrix = EvolutionMatrix(res_energies)

out = evo_matrix(temp, None , driven_probs)

In [6]:
def test_detailed_balance_two_level():
    """Verify detailed balance condition for 2-level system"""
    dtype = torch.float64
    E_K = torch.tensor([-2.0, 2.0], dtype=dtype)
    res_energies = mars.constants.unit_converter(E_K, "K_to_Hz")
    
    temp = torch.tensor(1.0, dtype=dtype)
    
    # Symmetric mean rates: k'_01 = k'_10 = 10 s⁻¹
    free_probs = torch.tensor([[0.0, 10.0], 
                               [10.0, 0.0]], dtype=dtype)
    
    evo = EvolutionMatrix(res_energies, thermal_balance_mode="symmetric")
    K = evo(temp, free_probs=free_probs)
    col_sums = K.sum(dim=-2)
    assert torch.allclose(col_sums, torch.zeros_like(col_sums), atol=1e-10), \
        f"Column sums not zero: {col_sums}"
    
    ΔE_01 = E_K[0] - E_K[1]  # -4 K
    expected_ratio_01 = torch.exp(-ΔE_01 / temp)  # exp(-(-4)/1) = exp(4) ≈ 54.6

    ratio_01 = K[0, 1] / K[1, 0]
    assert torch.allclose(ratio_01, expected_ratio_01, rtol=1e-5), \
        f"Detailed balance violated: got {ratio_01}, expected {expected_ratio_01}"
    
    n_eq = torch.exp(-E_K / temp)
    n_eq = n_eq / n_eq.sum()  # Normalize
    dn_dt = K @ n_eq
    assert torch.allclose(dn_dt, torch.zeros_like(dn_dt), atol=1e-10), \
        f"Equilibrium not stationary: {dn_dt}"

In [7]:
def test_temperature_dependence():
    """Verify rates become symmetric at high temperature"""
    dtype = torch.float64
    res_energies = mars.constants.unit_converter(
        torch.tensor([0.0, 10.0], dtype=dtype), "K_to_Hz"
    )
    
    free_probs = torch.tensor([[0.0, 5.0], [5.0, 0.0]], dtype=dtype)
    evo = EvolutionMatrix(res_energies, thermal_balance_mode="symmetric")
    
    K_low = evo(torch.tensor(1.0, dtype=dtype), free_probs=free_probs)
    assert K_low[0, 1] > 10 * K_low[1, 0]  # 0←1 much faster than 1←0
    
    K_high = evo(torch.tensor(10000.0, dtype=dtype), free_probs=free_probs)
    ratio = K_high[0, 1] / K_high[1, 0]
    assert torch.abs(ratio - 1.0) < 0.01, f"High-T asymmetry: {ratio}"


In [8]:
def test_superoperator_population_block():
    """Verify EvolutionSuper correctly scales population-population block"""
    dtype = torch.float64
    res_energies = mars.constants.unit_converter(
        torch.tensor([-1.0, 1.0], dtype=dtype), "K_to_Hz"
    )
    
    free_superop = torch.zeros(4, 4, dtype=dtype)
    free_superop[0, 3] = 5.0
    free_superop[3, 0] = 0.1
    
    H = torch.diag(res_energies)
    
    evo_super = EvolutionSuper(res_energies, thermal_balance_mode="symmetric")
    R = evo_super(temp=torch.tensor(1.0, dtype=dtype), 
                  H=H.unsqueeze(0), 
                  free_superop=free_superop.unsqueeze(0))
    
    R = R.squeeze(0)
    
    # Extract population block (indices 0 and 3 for 2-level system)
    pop_block = R[[0, 3]][:, [0, 3]]
    
    # Should satisfy detailed balance with energies
    E_K = torch.tensor([-1.0, 1.0], dtype=dtype)
    expected_ratio = torch.exp(-(E_K[0] - E_K[1]) / 1.0)  # exp(-(-2)) = exp(2)
    actual_ratio = pop_block[0, 1] / pop_block[1, 0]  # Rate 0←1 / 1←0
    
    assert torch.allclose(actual_ratio, expected_ratio.to(torch.complex128), rtol=1e-5), \
        f"Superoperator detailed balance failed: {actual_ratio} vs {expected_ratio}"

In [9]:
test_detailed_balance_two_level()
test_temperature_dependence()
test_superoperator_population_block()